# Timeseries plots and analysis

Based on data that has passed quality checks

In [ ]:
# Imports
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pingouin as pg

import blue_heart_data_handling as bhdh
from blue_heart_data_handling import get_gauge_data, DataTypeOptions, GaugeTypes, GaugeParameters

In [ ]:
# General settings

# Whether to save plots
save_flag = True

# Gauge index
path_index_telemetry_data = '../index_telemetry_data.csv'
df_index = pd.read_csv(path_index_telemetry_data)

# Folder to save plots in
path_output_folder = 'outputs/6_timeseries_plots'
if not os.path.isdir(path_output_folder):
    os.makedirs(path_output_folder)

# Plot settings
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 10
})

In [ ]:
# Function to plot timeseries on single plot

def plot_gauge_timeseries_data(
        gauge_dfs: dict[str, dict[GaugeParameters | str, pd.DataFrame]],
        plot_parameter: GaugeParameters | str,
        y_axis_label: str,
        num_gauges_limit: int | None = None,
        fig_width: float = 10,
        fig_height: float = 3,
):
    """
    Plot timeseries data from gauges

    Parameters
    gauge_dfs: Dictionary containing gauge parameter dataframes
    plot_parameter (GaugeParameters | str): Parameter to plot (second level key in gauge_dfs)
    y_axis_label (str): Y axis label corresponding to the plot parameter
    num_gauges_limit (int | None): Maximum number of gauges to plot
    fig_width (float): Figure width
    fig_height (float): Figure height
    """

    # Create plot
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    # Filter specified number of gauges, just based on order in dictionary (alphabetical if based on gauge index csv)
    if num_gauges_limit:
        gauge_dfs = dict(list(gauge_dfs.items())[:num_gauges_limit])

    # X axis range
    x_min = min(
        gauge_dfs[location][plot_parameter].index.min()
        for location in gauge_dfs
    )
    x_max = max(
        gauge_dfs[location][plot_parameter].index.max()
        for location in gauge_dfs
    )

    # Name of column to plot
    if isinstance(plot_parameter, GaugeParameters):
        plot_column = plot_parameter.value
    else:
        plot_column = plot_parameter

    # Add data to plot
    for location in gauge_dfs:
        df = gauge_dfs[location][plot_parameter]
        x = df.index.tz_localize(None)  # remove timezone
        y = df[plot_column]
        ax.plot(x, y, linewidth=0.8,)
    
    # Sort plot layout, labels etc.
    ax.set_xlabel('Date')
    ax.set_ylabel(y_axis_label)
    ax.set_xlim(x_min, x_max)
    ax.grid(False)
    plt.tight_layout()

    return fig

In [ ]:
# Function to identify gauge type from gauge name

def gauge_type_from_name(
        gauge_name: str,
        df_index: pd.DataFrame
    ):

    """Identify gauge type from gauge name
    
    Parameters:
        gauge_name (str): Name (location) of gauge
        df_index (pd.DataFrame): Gauge index dataframe

    Returns:
        str: Gauge type
    """
    
    return df_index[df_index['locationName'] == gauge_name]['gaugeType'].unique()[0]

## Highway gully water levels

In [ ]:
# Load data (all highway gullies)

dfs_gullies = get_gauge_data(
    data_type_option=DataTypeOptions.Quality1,
    filter_gauge_types=[GaugeTypes.HighwayGully],
    filter_gauge_parameters=[GaugeParameters.WaterLevelMM],
)

In [ ]:
# Plot data from 10 gauges on a single plot
fig = plot_gauge_timeseries_data(
        gauge_dfs=dfs_gullies,
        plot_parameter=GaugeParameters.WaterLevelMM,
        y_axis_label='Water level [mm]',
        num_gauges_limit=10
)

plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Highway gullies sample water levels.png', dpi=600, bbox_inches='tight')

plt.close(fig)


In [ ]:
# Calculate summary statistics (based on all highway gullies)

percentiles_1st = []
minima = []
ranges = []
percentile_ranges_1_99 = []
parameter = GaugeParameters.WaterLevelMM
for location in dfs_gullies.keys():
    # Get data for location
    series_parameter = dfs_gullies[location][parameter][parameter.value]
    # Calculate 1st percentile
    percentiles_1st.append(series_parameter.quantile(0.01))
    # Calculate minimum
    minima.append(series_parameter.min())
    # Calculate range
    ranges.append(series_parameter.max() - series_parameter.min())
    # Calculate 1st-99th percentile range
    percentile_ranges_1_99.append(series_parameter.quantile(0.99) - series_parameter.quantile(0.01))

print(f'Minimum 1st percentile level at a single gully {round(np.min(percentiles_1st), 0)} mm')
print(f'Maximum 1st percentile level at a single gully {round(np.max(percentiles_1st), 0)} mm')

print(f'\nMinimum minimum level at a single gully {round(np.min(minima), 0)} mm')
print(f'Maximum minimum level at a single gully {round(np.max(minima), 0)} mm')

print(f'\nMinimum range at a single gully {round(np.min(ranges), 0)} mm')
print(f'Maximum range at a single gully {round(np.max(ranges), 0)} mm')

print(f'\nMinimum 1st–99th percentile range at a single gully {round(np.min(percentile_ranges_1_99), 0)} mm')
print(f'Maximum 1st–99th percentile range at a single gully {round(np.max(percentile_ranges_1_99), 0)} mm')

## Borehole water levels

In [ ]:
# Load data (all boreholes)

dfs_boreholes = get_gauge_data(
    data_type_option=DataTypeOptions.Quality1,
    filter_gauge_types=[GaugeTypes.Borehole],
    filter_gauge_parameters=[GaugeParameters.WaterLevelM, GaugeParameters.DepthToWater],
)

In [ ]:
# Calculate deviation from minimum observed at each gauge

name_calculated_parameter = 'Water level deviation from gauge minimum [m]'

for location in dfs_boreholes.keys():
    parameter = list(dfs_boreholes[location].keys())[0]
    series_parameter = dfs_boreholes[location][parameter][parameter.value]
    if parameter == GaugeParameters.WaterLevelM:
        level_min = series_parameter.min()
        series_calculated = series_parameter - level_min
    elif parameter == GaugeParameters.DepthToWater:
        level_min = series_parameter.max()
        series_calculated = level_min - series_parameter
    else:
        series_calculated = series_parameter * np.nan
    df_calculated = series_calculated.to_frame().rename(
        columns={parameter.value: name_calculated_parameter})
    dfs_boreholes[location][name_calculated_parameter] = df_calculated

In [ ]:
# Plot data from all boreholes on a single plot
fig = plot_gauge_timeseries_data(
        gauge_dfs=dfs_boreholes,
        plot_parameter=name_calculated_parameter,
        y_axis_label='Water level [m]'
)

plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Borehole water levels.png', dpi=600, bbox_inches='tight')

plt.close(fig)

In [ ]:
# Calculate summary statistics (based on all borehole gauges)

# Range of depths captured at all gauges with at least one year of data
count = 0
level_ranges_m = []
for location in dfs_boreholes.keys():
    parameter = list(dfs_boreholes[location].keys())[0]
    series_parameter = dfs_boreholes[location][parameter][parameter.value]
    duration = series_parameter.index.max() - series_parameter.index.min()
    if duration > pd.Timedelta(days=365):
        count += 1
        level_range = np.abs(series_parameter.max() - series_parameter.min())
        level_ranges_m.append(level_range)

print(f'{count} borehole gauges with data spanning at least 365 days')
print(f'Minimum measurement range at a single borehole {round(min(level_ranges_m), 2)} m')
print(f'Maximum measurement range at a single borehole {round(max(level_ranges_m), 2)} m')

## Sewer water levels

In [ ]:
# Load data (all sewer levels)

dfs_sewers = get_gauge_data(
    data_type_option=DataTypeOptions.Quality1,
    filter_gauge_types=[GaugeTypes.Sewer],
    filter_gauge_parameters=[GaugeParameters.SewerLevel],
)


In [ ]:
# Plot data from 5 gauges on a single plot
fig = plot_gauge_timeseries_data(
        gauge_dfs=dfs_sewers,
        plot_parameter=GaugeParameters.SewerLevel,
        y_axis_label='Water level [mm]',
        num_gauges_limit=5
)

plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Sewer sample water levels.png', dpi=600, bbox_inches='tight')

plt.close(fig)

## Fluvial water levels

In [ ]:
# Load data (all fluvial levels)

dfs_fluvial = get_gauge_data(
    data_type_option=DataTypeOptions.Quality1,
    filter_gauge_types=[GaugeTypes.Fluvial],
    filter_gauge_parameters=[GaugeParameters.WaterLevelM, GaugeParameters.WaterDepth],
)

In [ ]:
# Unify parameters with different names

name_calculated_parameter = 'Water level [mAOD]'

for location in dfs_fluvial.keys():
    parameter = list(dfs_fluvial[location].keys())[0]
    series_parameter = dfs_fluvial[location][parameter][parameter.value]
    df_calculated = series_parameter.to_frame().rename(
        columns={parameter.value: name_calculated_parameter})
    dfs_fluvial[location][name_calculated_parameter] = df_calculated

In [ ]:
# Plot data from 10 gauges on a single plot
fig = plot_gauge_timeseries_data(
        gauge_dfs=dfs_fluvial,
        plot_parameter=name_calculated_parameter,
        y_axis_label='Water level [m]',
        num_gauges_limit=10
)

plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Fluvial sample water levels.png', dpi=600, bbox_inches='tight')

plt.close(fig)

In [ ]:
# Calculate summary statistics (based on all fluvial gauges)

percentiles_1st = []
minima = []
ranges = []
percentile_ranges_1_99 = []
for location in dfs_fluvial.keys():
    # Get data for location
    series_parameter = dfs_fluvial[location][name_calculated_parameter][name_calculated_parameter]
    # Calculate 1st percentile
    percentiles_1st.append(series_parameter.quantile(0.01))
    # Calculate minimum
    minima.append(series_parameter.min())
    # Calculate range
    ranges.append(series_parameter.max() - series_parameter.min())
    # Calculate 1st-99th percentile range
    percentile_ranges_1_99.append(series_parameter.quantile(0.99) - series_parameter.quantile(0.01))

print(f'Minimum 1st percentile level at a single fluvial gauge {round(np.min(percentiles_1st), 2)} m')
print(f'Maximum 1st percentile level at a single fluvial gauge {round(np.max(percentiles_1st), 2)} m')

print(f'\nMinimum minimum level at a single fluvial gauge {round(np.min(minima), 2)} m')
print(f'Maximum minimum level at a single fluvial gauge {round(np.max(minima), 2)} m')

print(f'\nMinimum range at a single fluvial gauge {round(np.min(ranges), 2)} m')
print(f'Median range at a single fluvial gauge {round(np.median(ranges), 2)} m')
print(f'Maximum range at a single fluvial gauge {round(np.max(ranges), 2)} m')

print(f'\nMinimum 1st–99th percentile range at a single fluvial gauge {round(np.min(percentile_ranges_1_99), 2)} m')
print(f'Median 1st–99th percentile range at a single fluvial gauge {round(np.median(percentile_ranges_1_99), 2)} m')
print(f'Maximum 1st–99th percentile range at a single fluvial gauge {round(np.max(percentile_ranges_1_99), 2)} m')

## Air temperature

In [ ]:
# Load data (all air temperature)

dfs_temperature_air = get_gauge_data(
    data_type_option=DataTypeOptions.Quality1,
    filter_gauge_parameters=[GaugeParameters.AirTemperature],
)

In [ ]:
# Plot data from 10 gauges on a single plot
fig = plot_gauge_timeseries_data(
        gauge_dfs=dfs_temperature_air,
        plot_parameter=GaugeParameters.AirTemperature,
        y_axis_label='Air temperature [°C]',
        num_gauges_limit=10
)

plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Air temperature sample data.png', dpi=600, bbox_inches='tight')

plt.close(fig)

In [ ]:
# Calculate summary statistics (based on all air temperature data)

percentile_ranges_1_99 = []
parameter = GaugeParameters.AirTemperature
for location in dfs_temperature_air.keys():
    # Get data for location
    series_parameter = dfs_temperature_air[location][parameter][parameter.value]

    # Determine number of data points expected in a year of data (only include if at least 95% of this value)
    timesteps_intentional = bhdh.get_intentional_timesteps(series_parameter.to_frame())
    timesteps_intentional_max = (
                    bhdh.get_max_intentional_timestep_minutes(timesteps_intentional)
                    )
    min_data_points_required = 365 * 24 * 60 / timesteps_intentional_max * 0.95
    count_data_points = len(series_parameter)
    if count_data_points < min_data_points_required:
        print(f' Only {count_data_points} data points at {location}, expected at least {min_data_points_required} in a year -> omitted from summary stats.')
        continue

    # Calculate 1st-99th percentile range
    percentile_ranges_1_99.append(series_parameter.quantile(0.99) - series_parameter.quantile(0.01))


print(f'\nMinimum 1st-99th percentile range at a air temperature gauge {round(np.min(percentile_ranges_1_99), 1)} °C')
print(f'Median 1st-99th percentile range at a air temperature gauge {round(np.median(percentile_ranges_1_99), 1)} °C')
print(f'Maximum 1st-99th percentile range at a air temperature gauge {round(np.max(percentile_ranges_1_99), 1)} °C')


In [ ]:
# Calculate summary statistics: Intra class correlation coefficient

# Base on hourly data, as this is the longest timestep used across the gauges
dfs_temperature_air_resampled = {
    location: dfs_temperature_air[location][parameter][parameter.value].resample('1h').mean()
    for location in dfs_temperature_air.keys()
}

# Combine in single dataframe
df_temperature_air_combined = pd.concat(dfs_temperature_air_resampled, axis=1, sort=True)

# Limit to timesteps with data from all gauges
df_temperature_air_combined.dropna(
    thresh=int(0.9 * df_temperature_air_combined.shape[1]), inplace=True)

# Calculate inter-gauge standard deviation
std_dev = df_temperature_air_combined.std(axis=1)

print(f'Minimum inter-gauge standard deviation = {round(std_dev.min(), 2)}')
print(f'Mean inter-gauge standard deviation = {round(std_dev.mean(), 2)}')
print(f'Maximum inter-gauge standard deviation = {round(std_dev.max(), 2)}')

# Change dataframe to long format
df_temperature_air_combined = (
    df_temperature_air_combined
        .reset_index()
        .melt(
            id_vars='timestamp',
            var_name='Gauge',
            value_name=parameter.value
        )
    )

# Calculate intra class correlation coefficient
icc_results = pg.intraclass_corr(
    data=df_temperature_air_combined,
    targets='timestamp',     # timestamps
    raters='Gauge',      # gauges
    ratings=parameter.value,
    nan_policy='omit'
).set_index('Type')

# Use ICC(C,1) value
icc = icc_results.loc['ICC(C,1)']['ICC']

print(f'\nIntra class correlation coefficient = {round(icc, 2)}')



## Water temperature

In [ ]:
# Load data (all water temperature)

dfs_temperature_water = get_gauge_data(
    data_type_option=DataTypeOptions.Quality1,
    filter_gauge_parameters=[GaugeParameters.WaterTemperature],
)

In [ ]:
# Plot data from 10 gauges on a single plot
fig = plot_gauge_timeseries_data(
        gauge_dfs=dfs_temperature_water,
        plot_parameter=GaugeParameters.WaterTemperature,
        y_axis_label='Water temperature [°C]',
        num_gauges_limit=10
)

plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Water temperature sample data.png', dpi=600, bbox_inches='tight')

plt.close(fig)

In [ ]:
# Calculate summary statistics (borehole and fluvial temperatures separately)

percentile_ranges_1_99_boreholes = []
percentile_ranges_1_99_fluvial = []
parameter = GaugeParameters.WaterTemperature
for location in dfs_temperature_water.keys():
    # Get data for location
    series_parameter = dfs_temperature_water[location][parameter][parameter.value]

    # Determine number of data points expected in a year of data (only include if at least 95% of this value)
    timesteps_intentional = bhdh.get_intentional_timesteps(series_parameter.to_frame())
    timesteps_intentional_max = (
                    bhdh.get_max_intentional_timestep_minutes(timesteps_intentional)
                    )
    min_data_points_required = 365 * 24 * 60 / timesteps_intentional_max * 0.95
    count_data_points = len(series_parameter)
    if count_data_points < min_data_points_required:
        print(f' Only {count_data_points} data points at {location}, expected at least {min_data_points_required} in a year -> omitted from summary stats.')
        continue

    # Calculate 1st-99th percentile range
    gauge_type = gauge_type_from_name(location, df_index)
    percentile_range_1_99 = series_parameter.quantile(0.99) - series_parameter.quantile(0.01)
    if gauge_type == GaugeTypes.Borehole.value:
        percentile_ranges_1_99_boreholes.append(series_parameter.quantile(0.99) - series_parameter.quantile(0.01))
    elif gauge_type == GaugeTypes.Fluvial.value:
        percentile_ranges_1_99_fluvial.append(series_parameter.quantile(0.99) - series_parameter.quantile(0.01))


print(f'\nMinimum 1st-99th percentile range at a borehole water temperature gauge {round(np.min(percentile_ranges_1_99_boreholes), 1)} °C')
print(f'Median 1st-99th percentile range at a borehole water temperature gauge {round(np.median(percentile_ranges_1_99_boreholes), 1)} °C')
print(f'Maximum 1st-99th percentile range at a borehole water temperature gauge {round(np.max(percentile_ranges_1_99_boreholes), 1)} °C')    


print(f'\nMinimum 1st-99th percentile range at a fluvial water temperature gauge {round(np.min(percentile_ranges_1_99_fluvial), 1)} °C')
print(f'Median 1st-99th percentile range at a fluvial water temperature gauge {round(np.median(percentile_ranges_1_99_fluvial), 1)} °C')
print(f'Maximum 1st-99th percentile range at a fluvial water temperature gauge {round(np.max(percentile_ranges_1_99_fluvial), 1)} °C')


In [ ]:
# Calculate summary statistics: Intra class correlation coefficient and inter-gauge standard deviation (fluvial gauges)

# Base on hourly data, as this is the longest timestep used across the gauges
dfs_temperature_water_resampled = {
    location: dfs_temperature_water[location][parameter][parameter.value].resample('1h').mean()
    for location in dfs_temperature_water.keys() if gauge_type_from_name(location, df_index) == GaugeTypes.Fluvial.value
}

# Combine in single dataframe
df_temperature_water_combined = pd.concat(dfs_temperature_water_resampled, axis=1, sort=True)

# Limit to timesteps with data from all gauges
df_temperature_water_combined.dropna(
    thresh=int(0.9 * df_temperature_water_combined.shape[1]), inplace=True)

# Calculate inter-gauge standard deviation
std_dev = df_temperature_water_combined.std(axis=1)

print(f'Minimum inter-gauge standard deviation = {round(std_dev.min(), 2)}')
print(f'Mean inter-gauge standard deviation = {round(std_dev.mean(), 2)}')
print(f'Maximum inter-gauge standard deviation = {round(std_dev.max(), 2)}')

# Change dataframe to long format
df_temperature_water_combined = (
    df_temperature_water_combined
        .reset_index()
        .melt(
            id_vars='timestamp',
            var_name='Gauge',
            value_name=parameter.value
        )
    )

# Calculate intra class correlation coefficient
icc_results = pg.intraclass_corr(
    data=df_temperature_water_combined,
    targets='timestamp',     # timestamps
    raters='Gauge',      # gauges
    ratings=parameter.value,
    nan_policy='omit'
).set_index('Type')

# Use ICC(C,1) value
icc = icc_results.loc['ICC(C,1)']['ICC']

print(f'\nIntra class correlation coefficient = {round(icc, 2)}')



## Rainfall

In [ ]:
# Load data (all rainfall)

dfs_rain = get_gauge_data(
    data_type_option=DataTypeOptions.Quality1,
    filter_gauge_types=[GaugeTypes.Rain]
)

In [ ]:
# Calculate daily totals for each gauge. Remove days with insufficient data.

name_calculated_parameter = 'Daily rainfall depth [mm]'
parameter = GaugeParameters.RainfallDepth
daily_timesteps_expected = 24 * 4       # 15 minute timesteps
completeness_threshold = 0.95   # Consider daily data incomplete below this

for location in dfs_rain.keys():
    # Get dataframe for location
    df = dfs_rain[location][parameter]

    # Get daily total and data counts
    daily_totals = df[parameter.value].resample('D').sum()
    daily_count = df[parameter.value].resample('D').count()

    # Calculate daily completeness and remove days with insufficient completeness
    completeness = daily_count / daily_timesteps_expected
    incomplete = completeness < completeness_threshold
    daily_totals = daily_totals.where(~incomplete)

    # Store in dictionary
    df_calculated = daily_totals.to_frame().rename(
        columns={parameter.value: name_calculated_parameter})
    df_incomplete = incomplete.to_frame().rename(
        columns={parameter.value: 'Incomplete'})
    dfs_rain[location][name_calculated_parameter] = df_calculated
    dfs_rain[location]['Incomplete'] = df_incomplete

In [ ]:
# Plot daily rainfall data with each gauge on separate subplot

# Number of gauges and middle position for axis labelling
gauges_count = len(dfs_rain)
gauges_middle_index = gauges_count // 2

# Create plot
fig, axes = plt.subplots(
    gauges_count,
    1,
    figsize=(10, 0.4 * gauges_count),
    sharex=True,
)

# X axis range
x_min = min(
    dfs_rain[location][name_calculated_parameter].index.min()
    for location in dfs_rain
)
x_max = max(
    dfs_rain[location][name_calculated_parameter].index.max()
    for location in dfs_rain
)

# Y axis range
y_min = 0
y_max = max(
    dfs_rain[location][name_calculated_parameter][name_calculated_parameter].max()
    for location in dfs_rain
)

# Add data to plot
for index, (ax, location) in enumerate(zip(axes, dfs_rain.keys())):

    # Plot rainfall
    daily_totals = dfs_rain[location][name_calculated_parameter][name_calculated_parameter]
    x = daily_totals.index.tz_localize(None)  # remove timezone
    y = daily_totals.values
    ax.bar(x, y, width=pd.Timedelta(days=1), color='blue')

    # Identify gaps, including before and after
    incomplete = dfs_rain[location]['Incomplete']['Incomplete']
    ax.axvspan(
        x_min,
        daily_totals.index.min().floor('D'),
        color='lightgrey',
        lw=0
    )
    ax.axvspan(
        daily_totals.index.max().ceil('D'),
        x_max,
        color='lightgrey',
        lw=0
    )
    gap_start = None
    for date, is_gap in incomplete.items():
        if is_gap and gap_start is None:
            gap_start = date
        elif not is_gap and gap_start is not None:
            ax.axvspan(
                gap_start,
                date,
                color='lightgrey',
                lw=0
            )
            gap_start = None
    if gap_start is not None:
        ax.axvspan(
            gap_start,
            incomplete.index[-1] + pd.Timedelta(days=1),
            color='lightgrey',
            lw=0
        )

    # Add location label
    ax.text(
        0.98, 0.92,
        location,
        transform=ax.transAxes,
        ha='right',
        va='top'
    )

    # Sort subplot layout, labels etc.
    ax.set_ylim(y_min, y_max * 1.05)
    ax.set_xlim(x_min, x_max)
    ax.set_yticks([0, 25])      # To avoid overlap at top with bottom of next plot
    if index == gauges_middle_index:
        ax.set_ylabel(name_calculated_parameter)
    ax.margins(x=0, y=0)

# Sort figure-level layout, labels etc.
axes[-1].set_xlabel('Date')
fig.subplots_adjust(hspace=0.07, wspace=0, top=1, bottom=0, left=0, right=1)

plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Rainfall daily data.png', dpi=600, bbox_inches='tight')

plt.close(fig)

In [ ]:
# Calculate summary statistics (based on all rain gauges)

intensity_maxes = []
rolling_day_maxes = []

for location in dfs_rain.keys():
    # Get dataframes for location (original timestep and daily)
    df = dfs_rain[location][parameter]

    # Identify peak 15 minute intensity captured
    intensity_max = df[parameter.value].max() * 4   # mm/hr, based on 15 min timestep
    intensity_maxes.append(intensity_max)

    # Calculate peak 24 hour total rainfall depth
    rolling_day_max = df[parameter.value].rolling('24h').sum().max()
    rolling_day_maxes.append(rolling_day_max)

print(f'Peak 15 minute rainfall intensity: {max(intensity_maxes)} mm / hr')
print(f'Peak 24 hour total rainfall: {max(rolling_day_maxes)} mm')

# Count gauges with a peak 15 minute intensity exceeding a specific threshold
intensity_threshold = 50
count = len([x for x in intensity_maxes if x > intensity_threshold])
print(f'Peak 15 minute rainfall intensity exceeding {intensity_threshold} mm/hr captured at {count} gauges ({round(100 * count / len(intensity_maxes), 1)} %)')

# Count gauges with a peak 24 hour total rainfall depth exceeding a specific threshold
depth_threshold = 50
count = len([x for x in rolling_day_maxes if x > depth_threshold])
print(f'Peak 24 hour total rainfall exceeding {depth_threshold} mm captured at {count} gauges ({round(100 * count / len(rolling_day_maxes), 1)} %)')


## Comparison of levels recorded at all gauges within a specified radius of a given point

In [ ]:
# Get names of gauges within specified radius of a given point

# Centre location and search radius
reference_lat = 50.808546
reference_long = 0.293948
radius_m = 1250

# Function to calculate distance between two points in metres
def haversine(
        lat1: float,
        lon1: float,
        lat2: float,
        lon2: float
    ):

    """Calculate distance between two points in metres
    
    Parameters:
        lat1 (float): Latitude of point 1
        lon1 (float): Longitude of point 1
        lat2 (float): Latitude of point 2
        lon2 (float): Longitude of point 2
    
    Returns:
        float: Distance in metres
 
    """

    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c * 1000

# Calculate distance for every gauge
df_index['Distance from reference point (m)'] = df_index.apply(
    lambda x: haversine(x['locationLatitude'], x['locationLongitude'], reference_lat, reference_long), axis=1
)

# Identify gauges within given radius of reference point
df_index_in_area = df_index[df_index['Distance from reference point (m)'] <= radius_m].copy()
print(f'Number of gauges in area: {len(df_index_in_area)}')

# Identify gauge types captured
gauge_type_counts = df_index_in_area['gaugeType'].value_counts()
print(f'\nGauge types in area: \n{gauge_type_counts}')

# Get list of gauge names
gauge_names = df_index_in_area['locationName'].tolist()

In [ ]:
# Load data for sample period (all level data collected within given area)

date_start = pd.Timestamp('2026-03-13', tz='UTC')
date_end   = pd.Timestamp('2026-03-14', tz='UTC')

filter_gauge_parameters = [
    GaugeParameters.SewerLevel,
    GaugeParameters.WaterDepth,
    GaugeParameters.WaterLevelMM,
    GaugeParameters.WaterLevelM
]

dfs_levels_in_area = get_gauge_data(
    data_type_option=DataTypeOptions.Quality1,
    filter_gauge_names=gauge_names,
    filter_gauge_parameters=filter_gauge_parameters,
    date_start=date_start,
    date_end=date_end
)

In [ ]:
# Calculate deviation from initial level observed at each gauge

name_calculated_parameter = 'Water level deviation from initial [m]'

for location in dfs_levels_in_area.keys():
    parameter = list(dfs_levels_in_area[location].keys())[0]
    series_parameter = dfs_levels_in_area[location][parameter][parameter.value]
    
    # Get initial level
    if len(series_parameter) > 0:
        level_first = series_parameter.iloc[0]
    else:
        level_first = np.nan
    series_calculated = series_parameter - level_first

    # Convert to metres if in mm
    if '[mm]' in parameter.value:
        series_calculated /= 1000

    # Store dataframe
    df_calculated = series_calculated.to_frame().rename(
        columns={parameter.value: name_calculated_parameter})
    dfs_levels_in_area[location][name_calculated_parameter] = df_calculated

In [ ]:
# Identify nearest rain gauge to centre of area with data for selected period
df_index_raingauges = df_index_in_area[df_index_in_area['gaugeType'] == 'Rain'].copy()
df_index_raingauges.sort_values('Distance from reference point (m)', inplace=True)

parameter = GaugeParameters.RainfallDepth
for index, row in df_index_raingauges.iterrows():
    location = row['locationName']
    print(f'Checking data from {location}')
    dfs_rain_in_area = get_gauge_data(
        data_type_option=DataTypeOptions.Quality1,
        filter_gauge_names=[location],
        filter_gauge_parameters=[parameter],
        date_start=date_start,
        date_end=date_end
    )
    df_rain = dfs_rain_in_area[location][parameter]
    if len(df_rain) > 0:
        print(f'Total rainfall depth: {df_rain['Rainfall Depth [mm]'].sum()} mm')
        print(f'Peak 15 min intensity: {df_rain['Rainfall Depth [mm]'].max()} mm')
        break
    else:
        print('   Insuficcient data for required period')




In [ ]:
# Plot water levels, colour coded by gauge type, with rainfall data shown as bars

# Gauge line styling
line_style_map = {
    'Borehole': 'b--',
    'Sewer': 'r-',
    'Fluvial': 'g-.',
    'Highway gully': 'k:'
}
line_width_map = {
    'Borehole': 2,
    'Sewer': 0.8,
    'Fluvial': 2,
    'Highway gully': 2
}
alpha_map = {
    'Borehole': 1,
    'Sewer': 0.6,
    'Fluvial': 1,
    'Highway gully': 1
}

# Create plot
fig, ax1 = plt.subplots(figsize=(10, 4))

# X axis range
x_min = date_start.tz_localize(None)
x_max = date_end.tz_localize(None)

# Y axis range


y1_min = min(
    dfs_levels_in_area[location][name_calculated_parameter][name_calculated_parameter].min()
    for location in dfs_levels_in_area
)
y1_max = max(
    dfs_levels_in_area[location][name_calculated_parameter][name_calculated_parameter].max()
    for location in dfs_levels_in_area
)

# Plot water levels on primary axis
for location in dfs_levels_in_area:
    gauge_type = gauge_type_from_name(location, df_index_in_area)
    df = dfs_levels_in_area[location][name_calculated_parameter]
    x = df.index
    y = df[name_calculated_parameter]
    ax1.plot(
        x, y,
        line_style_map[gauge_type],
        linewidth=line_width_map[gauge_type],
        label=gauge_type,
        alpha=alpha_map[gauge_type]
        )

# Plot rainfall on secondary axis
ax2 = ax1.twinx()
ax2.invert_yaxis()
ax2.bar(
    df_rain.index,
    df_rain['Rainfall Depth [mm]'],
    width=15 / (24 * 60),       # 15 minutes in days
    label='Rainfall',
    color='blue',
    alpha=0.8
)

# Sort plot layout, labels etc.
ax1.set_ylabel(f'Deviation from initial level [m]')
ax1.set_ylim(1.05 * y1_min, 1.1 * y1_max)
ax2.set_ylabel('15 min rainfall depth [mm]')
ax2.set_ylim(4 * df_rain['Rainfall Depth [mm]'].max(), 0)
ax1.set_xlabel('Date')
ax1.set_xlim(x_min, x_max)

# Add unique legend entries
handles, labels = ax1.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax1.legend(unique.values(), unique.keys(), loc='upper right')

plt.tight_layout()
plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Gauges in locality - deviation from initial.png', dpi=600, bbox_inches='tight')
    
plt.close(fig)